# Food Recognition & 3D Plate Scanning Research Pipeline
### Kaggle Cloud GPU Training: Clean Retraining of 43-Class Indian Food Classifier (Repaired Splits v2)

**Source of Truth Repository:** `https://github.com/VaibhavBajpaij/food-plat-scanner-ai-model.git`  
**Model Architecture:** MobileNetV3-Small (Official ImageNet-Pretrained Transfer Learning & Staged Bottleneck Fine-Tuning)  
**Repaired Dataset:** Enhanced Indian Food Classification (15,358 images across 43 regional Indian classes)  
**Manifest:** `data_manifests/indian_food_43class_repaired_manifest_v2.json` (Zero cross-split leakage, all 43 classes present in Train, Val, and Test)  

--- 
### ⚠️ Critical Pipeline & Compliance Rules
1. **Clean ImageNet Initialization:** Model weights are initialized strictly from `MobileNet_V3_Small_Weights.DEFAULT`, never from the contaminated historical food checkpoint.
2. **Strict Split Isolation:** The model trains **only** on the repaired `train` split (12,285 images), uses `val` (1,537 images) for early stopping and model selection, and evaluates the held-out `test` split (1,536 images) **exactly once** at the very end.
3. **Zero Cross-Split Leakage:** All 15,248 duplicate and near-duplicate groups are strictly isolated to single splits.
4. **Ephemeral Disk Storage & Output Recovery:** `/kaggle/working` is wiped when the notebook session closes. All final checkpoints, class mappings, evaluation reports, and model configurations are packaged into `/kaggle/working/kaggle_indian_43class_clean_v2_artifacts.zip` for download.
5. **Settings Checklist:** Ensure **Accelerator -> GPU (T4 x2 or P100)** and **Internet Access -> ON** in the Kaggle sidebar settings.

### Step 1: Clone or Update GitHub Repository (Source of Truth)

In [ ]:
import os
import sys
import subprocess

REPO_URL = "https://github.com/VaibhavBajpaij/food-plat-scanner-ai-model.git"
REPO_DIR = "/kaggle/working/food-plat-scanner-ai-model"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(f"Repository already exists at {REPO_DIR}. Resetting and syncing to latest origin/main...")
    subprocess.run(["git", "fetch", "origin", "main"], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "reset", "--hard", "origin/main"], cwd=REPO_DIR, check=True)
else:
    print(f"Cloning source of truth repository from {REPO_URL}...")
    subprocess.run(["git", "clone", "--branch", "main", REPO_URL, REPO_DIR], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print(f"Current Working Directory: {os.getcwd()}")

### Step 2: Check GPU Accelerator, Outgoing Internet & Available Disk Space

In [ ]:
import shutil
import urllib.request
import torch

print("=" * 85)
print("KAGGLE RUNTIME, GPU, INTERNET & STORAGE AUDIT")
print("=" * 85)

# 1. GPU Check
!nvidia-smi
cuda_available = torch.cuda.is_available()
print(f"\nPyTorch Version:         {torch.__version__}")
print(f"CUDA Available:          {cuda_available}")
if cuda_available:
    device_name = torch.cuda.get_device_name(0)
    total_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Active GPU Accelerator:  {device_name}")
    print(f"Total Dedicated VRAM:    {total_mem_gb:.2f} GB")
else:
    print("WARNING: GPU accelerator is NOT active! Navigate to Settings -> Accelerator -> GPU T4 x2 or P100.")

# 2. Internet Connectivity Check
try:
    with urllib.request.urlopen("https://huggingface.co", timeout=5) as response:
        print("Internet Connectivity:   ONLINE (Direct outbound HTTP access verified)")
except Exception as e:
    print(f"WARNING: Internet access test failed: {e}")
    print("Please toggle Internet -> ON in the Kaggle right-hand settings panel!")

# 3. Disk Space Check
total_disk, used_disk, free_disk = shutil.disk_usage("/kaggle/working")
print(f"Disk Space (/kaggle/working): Total={total_disk/(1024**3):.1f} GB | Free={free_disk/(1024**3):.1f} GB")
assert free_disk / (1024**3) > 4.0, "Error: Less than 4 GB disk space available!"
print("=" * 85)

### Step 3: Install Required Dependencies (Minimal Footprint)

In [ ]:
print("Installing dependencies for Parquet reading, fast decoding, and image processing...")
!pip install -q pyarrow fastparquet Pillow torchvision
print("Dependencies installed successfully.")

### Step 4: Ensure Dataset Shards & Extracted Images Are Present
Checks if dataset images already exist at `/kaggle/working/datasets/processed/indian_food_43class`.
If not present, downloads the 5 Parquet shards and extracts the authentic Indian food images.

In [ ]:
import os
from src.data.resumable_downloader import EnhancedIndianFoodAdapter
from src.data.parquet_food_loader import extract_and_prepare_43class_manifest

image_processed_dir = "/kaggle/working/datasets/processed/indian_food_43class"
parquet_raw_dir = "/kaggle/working/datasets/raw/sohlhealth_indian_food"

# Check if images are already extracted and ready
already_extracted = os.path.exists(image_processed_dir) and len(os.listdir(image_processed_dir)) >= 3

if already_extracted:
    print(f"Images already present at {image_processed_dir}! Skipping download and extraction.")
else:
    print("Dataset images not found in session working directory. Starting acquisition...")
    # 1. Download shards
    adapter = EnhancedIndianFoodAdapter(staging_dir=parquet_raw_dir)
    success = adapter.acquire(resume=True)
    assert success, "Fatal: Parquet shard acquisition failed!"
    
    # 2. Extract images
    os.makedirs("/kaggle/working/data_manifests", exist_ok=True)
    manifest_temp_path = "/kaggle/working/data_manifests/temp_extracted_manifest.json"
    extract_res = extract_and_prepare_43class_manifest(
        parquet_dir=parquet_raw_dir,
        output_image_dir=image_processed_dir,
        output_manifest_path=manifest_temp_path
    )
    print(f"Extracted {extract_res['total_images']:,} images into {image_processed_dir}.")

print("Dataset preparation verified.")

### Step 5: Verify Repaired 43-Class Manifest, Label Mapping & Zero Split Leakage
Verifies that `data_manifests/indian_food_43class_repaired_manifest_v2.json` exists,
confirms that 100% of referenced images are present, verifies 43 canonical labels, and asserts zero leakage.

In [ ]:
import json
from src.classification.retrain_clean_43class import (
    verify_label_mapping,
    verify_duplicate_group_isolation,
    verify_dataset_paths,
    CANONICAL_43_FOOD_CLASSES
)

manifest_v2_path = "data_manifests/indian_food_43class_repaired_manifest_v2.json"
assert os.path.exists(manifest_v2_path), f"Repaired manifest v2 not found at {manifest_v2_path}! Run 'git pull' in Step 1."

with open(manifest_v2_path, "r", encoding="utf-8") as f:
    man_data = json.load(f)

print("=" * 85)
print("PRE-FLIGHT AUDIT: REPAIRED MANIFEST INTEGRITY")
print("=" * 85)
print(f"Dataset:        {man_data.get('dataset_name')}")
print(f"Total Images:   {man_data.get('total_images'):,}")
print(f"Split Summary:  {man_data.get('split_summary')}")

# 1. Label Mapping
class_to_idx, idx_to_class = verify_label_mapping(man_data["classes"])
print(f"[PASS] Verified {len(class_to_idx)} canonical Indian food classes.")

# 2. Duplicate Group Isolation
iso = verify_duplicate_group_isolation(man_data["images"])
print(f"[PASS] Verified {iso['total_groups']:,} duplicate groups with zero cross-split leakage.")

# 3. Comprehensive Physical File Verification
all_ok, checked, missing, sample_missing = verify_dataset_paths(
    man_data["images"],
    base_dir="/kaggle/working",
    splits_to_verify=["train", "val"]
)
if not all_ok:
    print(f"[FAIL] Missing {missing:,} of {checked:,} images! Sample missing: {sample_missing}")
    assert False, "Cannot proceed: Some dataset image files are missing! Check Step 4."
else:
    print(f"[PASS] All {checked:,} referenced training and validation images physically exist on disk!")
print("=" * 85)

### Step 6: Execute Complete Scientific Unit & Integrity Test Suite (38 Tests)

In [ ]:
!python -m src.utils.run_unit_tests

### Step 7: Clean GPU Retraining on Repaired 43-Class Splits
- MobileNetV3-Small starting strictly from official ImageNet weights (`models.MobileNet_V3_Small_Weights.DEFAULT`).
- Trains strictly on repaired `train` split (12,285 images).
- Uses `val` split (1,537 images) for early stopping and model selection.
- Held-out `test` split (1,536 images) evaluated **strictly once** after training finishes.
- Automatically packages checkpoints, config, and metrics report into `/kaggle/working/kaggle_indian_43class_clean_v2_artifacts.zip`.

In [ ]:
!python -m src.classification.retrain_clean_43class \
    --manifest data_manifests/indian_food_43class_repaired_manifest_v2.json \
    --checkpoint-out models/checkpoints/mobilenet_v3_indian_43class_clean_v2_best.pth \
    --report-out outputs/indian_43class_clean_training_report.json \
    --epochs-head 5 \
    --epochs-finetune 10 \
    --batch-size 32

### Step 8: Verify Artifacts Package & Download Deliverables
Confirms the final checkpoint, validation report, and packaging ZIP in `/kaggle/working`.

In [ ]:
import os
import json
import zipfile
import torch

zip_path = "/kaggle/working/kaggle_indian_43class_clean_v2_artifacts.zip"
if not os.path.exists(zip_path):
    # Fallback path if inside repo
    alt_zip = "outputs/kaggle_indian_43class_clean_v2_artifacts.zip"
    if os.path.exists(alt_zip):
        import shutil
        shutil.copyfile(alt_zip, zip_path)

assert os.path.exists(zip_path), f"Artifacts ZIP not found at {zip_path}! Check Step 7 training output."
zip_size_mb = os.path.getsize(zip_path) / (1024**2)

print("=" * 85)
print("TRAINING COMPLETE & ARTIFACT PACKAGE VERIFIED")
print("=" * 85)
print(f"Artifact ZIP Path: {zip_path} ({zip_size_mb:.2f} MB)")

# List contents of ZIP
with zipfile.ZipFile(zip_path, 'r') as zf:
    print("\nPackaged Artifacts:")
    for info in zf.infolist():
        print(f"  - {info.filename:45s} ({info.file_size / (1024**2):.2f} MB)")

# Display Training and Test Evaluation Summary
report_path = "outputs/indian_43class_clean_training_report.json"
if os.path.exists(report_path):
    with open(report_path, "r", encoding="utf-8") as f:
        rep = json.load(f)
    test_m = rep.get("held_out_test_metrics", {})
    print("\n--- FINAL HELD-OUT TEST EVALUATION SUMMARY ---")
    print(f"Accuracy:          {test_m.get('accuracy', 0.0):.2%} (95% CI: [{test_m.get('accuracy_ci_95', [0,0])[0]:.2%}, {test_m.get('accuracy_ci_95', [0,0])[1]:.2%}])")
    print(f"Balanced Accuracy: {test_m.get('balanced_accuracy', 0.0):.2%}")
    print(f"Macro-F1:          {test_m.get('macro_f1', 0.0):.4f}")
    print(f"Weighted-F1:       {test_m.get('weighted_f1', 0.0):.4f}")
    print(f"Total Test Images: {test_m.get('total_samples', 0):,} (Evaluated exactly once across all 43 classes)")
    print(f"Training Duration: {rep.get('training_duration_seconds', 0):.1f} seconds")

print("=" * 85)
print("ACTION REQUIRED: Go to the 'Output' tab on the right side of Kaggle and download:")
print("  -> 'kaggle_indian_43class_clean_v2_artifacts.zip'")
print("Place this zip file into your local project folder to complete the evaluation loop.")
print("=" * 85)